# 5. The five steps

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

One forward pass turns the four inputs from chapter 1 into vertices. The intermediate names come back in every later chapter. This chapter puts them in time order.

```text
v_shaped = v_template + shapedirs · betas
J        = J_regressor · v_shaped
v_posed  = v_shaped + posedirs · (R(pose) - I)     # 15 finger joints only
vertices = LBS(v_posed, J, pose, weights) + transl
joints   = FK(J, pose) + transl
```

```mermaid
flowchart TD
  betas --> v_shaped["v_shaped = template + shape"]
  template["v_template"] --> v_shaped
  v_shaped --> J["J = regressor × v_shaped"]
  pose["15 finger axis-angles"] --> feature["135 numbers = R - I"]
  feature --> v_posed["v_posed = v_shaped + pose correction"]
  v_shaped --> v_posed
  pose --> fk["forward kinematics"]
  orient["global_orient"] --> fk
  J --> fk
  fk --> lbs["linear blend skinning"]
  v_posed --> lbs
  weights["weights"] --> lbs
  lbs --> vertices["778 vertices"]
  fk --> joints["16 joints"]
  transl --> vertices
  transl --> joints
```

## What each step leaves behind

| Step | Result | What changes it |
|---|---|---|
| Template `v_template` | 778×3, the mean hand, metres | A constant in the model file |
| Shape `v_shaped` | The same 778×3 | `betas` |
| Rest joints `J` | 16×3 | `betas`. Pose does not edit `J` |
| Pose feature | 135 numbers | The rotations of the 15 finger joints. The wrist is not included |
| `v_posed` | Vertices plus the flesh correction | Shape and finger pose |
| Skinned `vertices` | The mesh you draw | Everything above, then `transl` |
| `joints` | The 16 points you align | `J`, the full pose, and `transl` |

Report the joints from the forward-kinematics line. Do not multiply the skinned vertices by `J_regressor` a second time. Once a vertex is shared by two bones, or a pose blend moves the surface, those two sets of points separate.

## Where this lives in the code

`OfficialMano.forward` in `official_mano.py` is this order. `toy_mano.py` runs the same arithmetic on 5 vertices and 4 joints. The next cell runs the toy once and lists the intermediate arrays. The same keys on the official model start in chapter 9.


In [2]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import toy_mano as toy

out = toy.forward()

for key, value in out.items():
    print(f"{key:<14} {np.shape(value)}")

wrist_only = np.zeros(12)
wrist_only[0] = np.deg2rad(30)
print("pose-feature norm, fingers at 0:", np.linalg.norm(toy.pose_feature(np.zeros(12))))
print("pose-feature norm, wrist only:  ", np.linalg.norm(toy.pose_feature(wrist_only)))
assert np.allclose(toy.pose_feature(np.zeros(12)), 0.0)
assert np.allclose(toy.pose_feature(wrist_only), 0.0)


vertices       (5, 3)
joints         (4, 3)
joints_rest    (4, 3)
v_shaped       (5, 3)
v_posed        (5, 3)
pose_feature   (27,)
pose-feature norm, fingers at 0: 0.0
pose-feature norm, wrist only:   0.0


The toy's units are arbitrary. They are not metres. The official model is in metres. The pose feature is 0 when every finger rotation is the identity, and rotating only the wrist leaves it at 0.

## Check

Say the five names in order: `v_shaped`, `J`, `v_posed`, `vertices`, `joints`. If you change only `global_orient` and leave the finger axis-angles at 0, the 135-D pose feature is all zeros.

The next chapter computes shape and expands the first line of the formula.
